# 02 — Cleaning the Primary Dataset

Turns the raw `spotify_tracks.csv` into tidy, normalized tables in `data/processed/`. Every filter here traces back to a specific issue found in `01_explore.ipynb`.

Output tables:
- `tracks` — one row per track (metadata + popularity)
- `audio_features` — one row per track (the 12 audio signals)
- `track_genres` — track ↔ genre bridge (many-to-many)
- `track_artists` — track ↔ artist bridge (many-to-many, with credit order)

## Load and filter

Drop rows that are unusable (missing name/artist) or physically impossible (tempo of 0, sub-30s or over-30min durations). These are extraction artifacts, not genuine outliers, and total well under 1% of rows.

In [8]:
import pandas as pd

raw = pd.read_csv("../data/raw/spotify_tracks.csv", index_col=0)

df = raw.dropna(subset=["track_id", "track_name", "artists"])
df = df[df["tempo"] > 0]
df = df[df["duration_ms"].between(30_000, 1_800_000)]

print(f"raw rows:     {len(raw):,}")
print(f"kept rows:    {len(df):,}")
print(f"dropped rows: {len(raw) - len(df):,}")

raw rows:     114,000
kept rows:    113,794
dropped rows: 206


## Genre bridge (before deduplication)

The raw file repeats a track once per genre. We capture every distinct track–genre pair *first*, so the many-to-many relationship is preserved before we collapse tracks to one row each.

In [9]:
track_genres = df[["track_id", "track_genre"]].drop_duplicates()

print(f"track–genre pairs: {len(track_genres):,}")
print(f"distinct genres:  {track_genres['track_genre'].nunique()}")

track–genre pairs: 113,345
distinct genres:  114


## Deduplicate to one row per track

Audio features are identical across a track's duplicate rows; popularity can differ by a point or two between playlist snapshots, so we keep the highest observed value. Sorting by popularity descending and keeping the first occurrence does exactly that.

In [10]:
audio_cols = [
    "danceability", "energy", "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
    "time_signature",
]

dedup = (
    df.sort_values("popularity", ascending=False)
      .drop_duplicates(subset="track_id", keep="first")
)

tracks = dedup[["track_id", "track_name", "album_name",
                "popularity", "duration_ms", "explicit"]]
audio_features = dedup[["track_id"] + audio_cols]

print(f"unique tracks: {len(tracks):,}")
print(f"audio rows:    {len(audio_features):,}  (should match)")

unique tracks: 89,539
audio rows:    89,539  (should match)


## Artist bridge (explode the artist string)

The `artists` field packs multiple credits into one `;`-separated string. We split it into individual artists, one row per track–artist pair, and record credit order (`position` 0 = primary artist).

In [11]:
track_artists = (
    dedup[["track_id", "artists"]]
    .assign(artist_name=lambda d: d["artists"].str.split(";"))
    .explode("artist_name")
)
track_artists["artist_name"] = track_artists["artist_name"].str.strip()

# Drop non-artists: empty credits from a trailing ";" and the literal "N/A"
# placeholder (a "no data" marker in the source, not a real artist).
placeholders = ["", "N/A"]
track_artists = track_artists[~track_artists["artist_name"].isin(placeholders)]

track_artists["position"] = track_artists.groupby("track_id").cumcount()
track_artists = track_artists[["track_id", "artist_name", "position"]]

print(f"track–artist pairs: {len(track_artists):,}")
print(f"distinct artists:  {track_artists['artist_name'].nunique():,}")

track–artist pairs: 122,992
distinct artists:  29,779


## Write the cleaned tables

In [12]:
from pathlib import Path

out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

tracks.to_csv(out / "tracks.csv", index=False)
audio_features.to_csv(out / "audio_features.csv", index=False)
track_genres.to_csv(out / "track_genres.csv", index=False)
track_artists.to_csv(out / "track_artists.csv", index=False)

print("wrote 4 tables to data/processed/")

wrote 4 tables to data/processed/
